In [ ]:
# ============================================================
# SETUP - Run this cell first
# ============================================================
!git clone https://github.com/tatipar/temporalgnn-nids.git
import sys
sys.path.append('/content/temporalgnn-nids/code/python')

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/nids-mitre/')

In [ ]:
!pip install torch_geometric

In [ ]:
import os
import numpy as np
import pandas as pd
import copy
import random
import json
import time

import torch
import torch.nn as nn

from torch_geometric.loader import DataLoader


In [ ]:
from utils.datasets   import NF_IDS_Dataset
from utils.models     import E_GraphSAGE
from utils.metrics    import calculate_metrics_gnn
from utils.training   import train_epoch, evaluate, find_optimal_threshold, set_seed, run_multiple_seeds
from utils.experiment import ExperimentManager, EarlyStopping, NumpyEncoder
from utils.visualization import save_plots

# Auxiliary

In [ ]:
ROOT_PATH = "./dataset_processed"

In [ ]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False)

Train size: 1998 | Val size: 428


In [ ]:
ROOT_DIR = "./results_earlystopping"


LOGS_DIR = os.path.join(ROOT_DIR, "logs")
PLOTS_DIR = os.path.join(ROOT_DIR, "plots")
MODELS_DIR = os.path.join(ROOT_DIR, "saved_models")


# Main

## Seeds

In [ ]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

#PROB_THRESHOLD = 0.5



Using device: cpu


In [ ]:
model_config = {
    "model_name": None,
    "type": "E_GraphSAGE",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE,
    },
    #"prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [ ]:
EXPERIMENT_NAME="EGraphSAGE_BiasOn"

exp_manager = ExperimentManager(log_file=os.path.join(LOGS_DIR, EXPERIMENT_NAME, f"run_metrics_{EXPERIMENT_NAME}.csv"), model_dir=os.path.join(MODELS_DIR, EXPERIMENT_NAME))

In [ ]:
run_multiple_seeds(
    model_class=E_GraphSAGE,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[42, 123, 777, 2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name=EXPERIMENT_NAME,
    json_dir=LOGS_DIR,
    plots_dir=PLOTS_DIR
)

 Starting Multi-Seed Run: SimpleMLP_BiasOn
   Seeds: [42, 123, 777, 2024, 99]
------------------------------------------------------------

Running seed 42 | run_id=SimpleMLP_BiasOn_seed42_20260219_181723

SimpleMLP_BiasOn_seed42
   Ep 1 | Loss: 0.2027 | Val Loss: 0.2297 | Val AUC-PR: 0.0904 (*)
   Ep 4 | Loss: 0.1982 | Val Loss: 0.2255 | Val AUC-PR: 0.1557 (*)
   Ep 5 | Loss: 0.1957 | Val Loss: 0.2244 | Val AUC-PR: 0.1590 (*)
   Ep 6 | Loss: 0.1949 | Val Loss: 0.2235 | Val AUC-PR: 0.1720 (*)
   Ep 7 | Loss: 0.1943 | Val Loss: 0.2230 | Val AUC-PR: 0.1728 (*)
   Ep 8 | Loss: 0.1940 | Val Loss: 0.2224 | Val AUC-PR: 0.1759 (*)
   Ep 10 | Loss: 0.1934 | Val Loss: 0.2221 | Val AUC-PR: 0.1776 (*)
   Ep 17 | Loss: 0.1930 | Val Loss: 0.2211 | Val AUC-PR: 0.1836 (*)
   Ep 20 | Loss: 0.1919 | Val Loss: 0.2210 | Val AUC-PR: 0.1738 
   Early Stopping in epoch 26
   
Optimal Threshold: 0.2123 (Max Recall @ Prec>=0.9)
   Final: Precision=0.9015 |  Recall=0.0190 | F1=0.0372 | F2=0.0236 | AUC-PR=0.183